In [ ]:
!pip install gTTS SpeechRecognition -q


In [ ]:
import time
from gtts import gTTS
from IPython.display import Audio, display, clear_output
from google.colab import output
import speech_recognition as sr
from base64 import b64decode

def falar(texto):
    if texto:
        tts = gTTS(text=texto, lang='pt-br')
        tts.save("response.mp3")
        display(Audio("response.mp3", autoplay=True))

def ouvir():
    js_code = """
    const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
    const b2text = blob => new Promise(resolve => {
      const reader = new FileReader()
      reader.onloadend = e => resolve(e.srcElement.result)
      reader.readAsDataURL(blob)
    })
    var record = time => new Promise(async resolve => {
      stream = await navigator.mediaDevices.getUserMedia({ audio: true })
      recorder = new MediaRecorder(stream)
      chunks = []
      recorder.ondataavailable = e => chunks.push(e.data)
      recorder.start()
      await sleep(time)
      recorder.onstop = async ()=>{
        blob = new Blob(chunks)
        text = await b2text(blob)
        resolve(text)
      }
      recorder.stop()
    })
    """
    display(output.eval_js(js_code))
    try:
        audio_data = output.eval_js('record(4000)')
        binary = b64decode(audio_data.split(','))
        with open('audio.wav', 'wb') as f: f.write(binary)
        recognizer = sr.Recognizer()
        with sr.AudioFile('audio.wav') as source:
            audio = recognizer.record(source)
            return recognizer.recognize_google(audio, language='pt-BR').lower()
    except:
        return ""


In [ ]:
import difflib

# Dicionário de respostas
responses = {
    "oi": "Olá! Como posso te ajudar hoje Aluno?",
    "olá": "Olá! Como posso te ajudar hoje Aluno?",
    "como vai": "Melhor que você, já que não preciso pagar contas.",
    "nome": "Me chamo Helpstudy!",
    "prova": "Se você não anotou, o desespero é real.",
    "bhaskara": "É a fórmula mágica: x = (-b ± √Δ) / 2a.",
    "estudar": "Focar nos estudos é o melhor caminho!",
    "obrigado": "De nada! Estude bastante.",
}

def buscar_melhor_resposta(texto_usuario):
    """Lógica para encontrar a resposta ou sugerir algo parecido"""
    texto_usuario = texto_usuario.lower().strip()

    # 1. Tenta busca direta (palavra dentro da frase)
    for chave in responses:
        if chave in texto_usuario:
            return responses[chave], None

    # 2. Se não achou, busca termos parecidos (Sugestão)
    chaves = list(responses.keys())
    sugestoes = difflib.get_close_matches(texto_usuario, chaves, n=1, cutoff=0.5)

    if sugestoes:
        return None, sugestoes[0] # Retorna None para a resposta e a palavra sugerida

    return "Sinto muito, não entendi. Pode repetir?", None


In [ ]:
def helpstudy_v4_executar():
    print("--- Helpstudy v4: Online ---")

    while True:
        print("\n📢 [F] Falar | [D] Digitar | [S] Sair")d
        opcao = input("Aguardando... ").lower().strip()

        if opcao == 's':
            falar("Até logo Welison!")
            break

        # Obtém o texto (Voz ou Teclado)
        user_input = ouvir() if opcao == 'f' else input("Sua dúvida: ")

        if not user_input or len(user_input) < 2:
            print("⚠️ Não captamos nada. Tente de novo.")
            continue

        # Processa a resposta usando a função da Célula 3
        resposta, sugestao = buscar_melhor_resposta(user_input)

        # Se houver uma sugestão em vez de resposta direta
        if sugestao:
            msg_sugestao = f"Você quis dizer '{sugestao}'?"
            print(f"🤖 {msg_sugestao}")
            falar(msg_sugestao)

            confirmar = input("Confirma? (S/N): ").lower().strip()
            if confirmar == 's':
                resposta = responses[sugestao]
            else:
                resposta = "Tudo bem. Como posso te ajudar então?"

        # Exibe e fala o resultado final
        clear_output(wait=True)
        print(f"🎙️ Você disse: {user_input}")
        print(f"🤖 Bot: {resposta}")
        falar(resposta)
        time.sleep(2)

# Inicia o programa
helpstudy_v4_executar()
f


📢 [F] Falar | [D] Digitar | [S] Sair


None

⚠️ Não captamos nada. Tente de novo.

📢 [F] Falar | [D] Digitar | [S] Sair
